# 🧪 Proyecto Final — Mini-Lakehouse con UCI Adult (PySpark)
**Entrega sugerida:** 2025-08-09 + 7 días (ajústalo en el enunciado).  
**Entorno:** Databricks Free Edition (hive_metastore, **tablas managed Delta**).  
**Dataset:** `dbfs:/databricks-datasets/adult/` (`adult.data`, `adult.test`, `adult.names`)

**Objetivo:** Construir un pipeline **Bronze → Silver → Gold** con **Delta Lake**:
- Bronze: ingesta cruda (todo como *string*), preservando el origen.
- Silver: limpieza, **casts**, derivadas, **deduplicación** (window).
- Gold: 3 tablas de **KPIs**.
- Operaciones Delta: `DESCRIBE HISTORY`, **time travel** y (si aplica) **`OPTIMIZE/ZORDER`** y **`VACUUM`**.

Completa las celdas marcadas **TODO** y ejecuta en orden.


## 0) Parámetros y Setup

In [0]:

# TODO: Reemplaza con tu identificador corto (solo letras/números/guiones bajos)
student = "TU_NOMBRE"

import re
student_slug = re.sub(r"[^a-zA-Z0-9_]", "_", student).lower()
db = f"bd_adult_{student_slug}"

print("Schema destino:", db)
spark.sql(f"CREATE DATABASE IF NOT EXISTS {db}")
spark.sql(f"USE {db}")

# Mostrar versión de Spark
spark.version


## 1) Bronze — Ingesta cruda desde `adult.data` + `adult.test`
Cargamos como **texto** y partimos por coma para ser robustos ante cabeceras/comentarios.  
**Nota:** Dejamos **todo en STRING** en Bronze. Añadimos `ingest_ts` y `ingest_file`.


In [0]:

from pyspark.sql.functions import *
from pyspark.sql.types import *
from datetime import datetime

adult_cols = [
  "age","workclass","fnlwgt","education","education_num",
  "marital_status","occupation","relationship","race","sex",
  "capital_gain","capital_loss","hours_per_week","native_country","income"
]

def load_adult_file(path):
    # Lee líneas de texto y filtra vacías/comentarios o cabecera accidental
    df_txt = (spark.read.format("text").load(path)
              .select(col("value").alias("line")))
    df_txt = df_txt.filter(col("line").isNotNull() & (length(trim(col("line"))) > 0))
    # Ignorar líneas que comienzan con '|' (comentarios)
    df_txt = df_txt.filter(~trim(col("line")).startswith("|"))
    # Ignorar cabecera si aparece (líneas que inician con 'age,workclass' o similar)
    df_txt = df_txt.filter(~lower(trim(col("line"))).startswith("age,workclass"))
    # Split por coma en 15 campos
    parts = split(col("line"), ",")
    # Asegurar al menos 15 columnas (algunas líneas pueden ser inválidas)
    df_raw = df_txt.filter(size(parts) >= 15).select([trim(parts[i]).alias(adult_cols[i]) for i in range(15)])
    # Añadir metadata de ingesta
    df_raw = (df_raw
              .withColumn("ingest_ts", current_timestamp())
              .withColumn("ingest_file", lit(path)))
    return df_raw

path_train = "dbfs:/databricks-datasets/adult/adult.data"
path_test  = "dbfs:/databricks-datasets/adult/adult.test"

bronze_df = load_adult_file(path_train).unionByName(load_adult_file(path_test))

# Inspección rápida
display(bronze_df.limit(10))
print("Registros totales (bronze_df):", bronze_df.count())

# Persistimos como Delta managed
(bronze_df.write
 .mode("overwrite")
 .format("delta")
 .saveAsTable(f"{db}.adult_bronze")
)

spark.sql(f"DESCRIBE HISTORY {db}.adult_bronze").show(truncate=False)
spark.table(f"{db}.adult_bronze").count()


## 2) Silver — Limpieza, tipificación, derivadas y deduplicación
**Transformaciones mínimas:**  
- Reemplazar `'?'` por `NULL` en categóricas clave (`workclass`, `occupation`, `native_country`).  
- Normalización: `trim`, `lower` (o `initcap` cuando convenga).  
- Arreglar `income` de `adult.test` removiendo el punto final (`regexp_replace(income, '\\.$', '')`).  
- **Casts** numéricos a `INT`: `age`, `fnlwgt`, `education_num`, `capital_gain`, `capital_loss`, `hours_per_week`.  
- Derivadas: `capital_net`, `age_bucket`, `work_hours_bucket`, `income_binary`.  
- **Reglas de calidad**: eliminar filas inválidas.  
- **Deduplicación** con ventana por clave compuesta (última por `ingest_ts`).  


In [0]:

from pyspark.sql.window import Window

bronze = spark.table(f"{db}.adult_bronze")

# Normalización y limpieza básica
norm = (bronze
    # Quitar punto final de income si existe (adult.test)
    .withColumn("income", regexp_replace(col("income"), "\\.$", ""))
    # Reemplazar '?' por NULL en categóricas
    .withColumn("workclass", when(col("workclass")=="?", None).otherwise(col("workclass")))
    .withColumn("occupation", when(col("occupation")=="?", None).otherwise(col("occupation")))
    .withColumn("native_country", when(col("native_country")=="?", None).otherwise(col("native_country")))
    # Normalización de cadenas
    .withColumn("workclass", lower(trim(col("workclass"))))
    .withColumn("education", lower(trim(col("education"))))
    .withColumn("marital_status", lower(trim(col("marital_status"))))
    .withColumn("occupation", lower(trim(col("occupation"))))
    .withColumn("relationship", lower(trim(col("relationship"))))
    .withColumn("race", lower(trim(col("race"))))
    .withColumn("sex", lower(trim(col("sex"))))
    .withColumn("native_country", initcap(trim(col("native_country"))))
    .withColumn("income", trim(lower(col("income"))))
)

# Tipificación y derivadas
silver_pre = (norm
    .withColumn("age_int", col("age").cast("int"))
    .withColumn("fnlwgt_int", col("fnlwgt").cast("int"))
    .withColumn("education_num_int", col("education_num").cast("int"))
    .withColumn("capital_gain_int", col("capital_gain").cast("int"))
    .withColumn("capital_loss_int", col("capital_loss").cast("int"))
    .withColumn("hours_per_week_int", col("hours_per_week").cast("int"))
    .withColumn("capital_net", col("capital_gain_int") - col("capital_loss_int"))
    .withColumn("age_bucket",
        when(col("age_int") < 25, "<25")
       .when((col("age_int") >= 25) & (col("age_int") <= 34), "25-34")
       .when((col("age_int") >= 35) & (col("age_int") <= 44), "35-44")
       .when((col("age_int") >= 45) & (col("age_int") <= 54), "45-54")
       .when((col("age_int") >= 55) & (col("age_int") <= 64), "55-64")
       .otherwise("65+"))
    .withColumn("work_hours_bucket",
        when(col("hours_per_week_int") < 20, "<20")
       .when((col("hours_per_week_int") >= 20) & (col("hours_per_week_int") <= 39), "20-39")
       .when(col("hours_per_week_int") == 40, "40")
       .when((col("hours_per_week_int") >= 41) & (col("hours_per_week_int") <= 59), "41-59")
       .otherwise("60+"))
    .withColumn("income_binary", when(col("income") == ">50k", lit(1)).otherwise(lit(0)))
)

# Reglas de calidad mínimas
silver_clean = (silver_pre
    .filter(col("age_int").isNotNull() & (col("age_int") > 0))
    .filter(col("hours_per_week_int").isNotNull() & (col("hours_per_week_int") >= 0))
    .filter(col("income").isNotNull())
)

# Deduplicación por clave compuesta (última por ingest_ts)
dup_key_cols = ["age","workclass","fnlwgt","education","education_num",
                "marital_status","occupation","relationship","race","sex",
                "capital_gain","capital_loss","hours_per_week","native_country","income"]

w = Window.partitionBy(*dup_key_cols).orderBy(col("ingest_ts").desc_nulls_last())
silver_dedup = (silver_clean
                .withColumn("rn", row_number().over(w))
                .filter(col("rn") == 1)
                .drop("rn"))

# Persistimos Silver
(silver_dedup
 .write
 .mode("overwrite")
 .format("delta")
 .saveAsTable(f"{db}.adult_silver")
)

display(silver_dedup.limit(10))
spark.sql(f"DESCRIBE HISTORY {db}.adult_silver").show(truncate=False)


## 3) Gold — Tablas de métricas/KPIs

### 3.1) `gold_income_by_demo` — tasa >50K por educación y sexo

In [0]:

spark.sql(f"""
CREATE OR REPLACE TABLE {db}.gold_income_by_demo AS
SELECT
  education,
  sex,
  COUNT(*) AS n,
  SUM(income_binary) AS high_income,
  AVG(CAST(income_binary AS DOUBLE)) AS high_income_rate,
  AVG(age_int) AS avg_age,
  AVG(hours_per_week_int) AS avg_hours_per_week
FROM {db}.adult_silver
GROUP BY education, sex
""")
)
display(spark.table(f"{db}.gold_income_by_demo").orderBy(col("high_income_rate").desc(), col("n").desc()).limit(20))


### 3.2) `gold_age_education_matrix` — matriz por age_bucket × education

In [0]:

spark.sql(f"""
CREATE OR REPLACE TABLE {db}.gold_age_education_matrix AS
SELECT
  age_bucket,
  education,
  COUNT(*) AS n,
  AVG(CAST(income_binary AS DOUBLE)) AS high_income_rate,
  AVG(hours_per_week_int) AS avg_hours_per_week
FROM {db}.adult_silver
GROUP BY age_bucket, education
""")
)
display(spark.table(f"{db}.gold_age_education_matrix").orderBy("age_bucket","education").limit(50))


### 3.3) `gold_country_profile` — perfil por país

In [0]:

spark.sql(f"""
CREATE OR REPLACE TABLE {db}.gold_country_profile AS
SELECT
  native_country,
  COUNT(*) AS n,
  AVG(CAST(income_binary AS DOUBLE)) AS high_income_rate,
  AVG(age_int) AS avg_age,
  AVG(hours_per_week_int) AS avg_hours_per_week,
  AVG(capital_net) AS avg_capital_net
FROM {db}.adult_silver
GROUP BY native_country
""")
)
display(spark.table(f"{db}.gold_country_profile").orderBy(col("n").desc()).limit(50))


## 4) Vistas (views)

In [0]:

# Vista persistente: segmentos de alto ingreso (threshold configurable)
spark.sql(f"""
CREATE OR REPLACE VIEW {db}.v_high_income_segments AS
SELECT *
FROM {db}.gold_income_by_demo
WHERE high_income_rate >= 0.5
""")
)

# Vista temporal: filas ingeridas hoy (si re-ejecutas hoy)
spark.sql(f"""
CREATE OR REPLACE TEMP VIEW v_ingest_today AS
SELECT *
FROM {db}.adult_silver
WHERE DATE(ingest_ts) = current_date()
""")
display(spark.sql("SELECT * FROM v_ingest_today LIMIT 20"))


## 5) Operaciones Delta — History, Time Travel, OPTIMIZE, VACUUM

In [0]:

# HISTORY de tablas clave
for t in ["adult_bronze","adult_silver","gold_income_by_demo","gold_age_education_matrix","gold_country_profile"]:
    print("\n=== HISTORY:", t, "===")
    spark.sql(f"DESCRIBE HISTORY {db}.{t}").show(truncate=False)


In [0]:

# TODO (time travel): simula un cambio "erróneo" en un subconjunto pequeño y consulta VERSION AS OF
# Ejemplo (REVISAR antes de ejecutar):
# spark.sql(f"""
#   UPDATE {db}.adult_silver
#   SET workclass = NULL
#   WHERE sex = 'female' AND education = 'bachelors' AND hours_per_week_int >= 40
#   LIMIT 50
# """)
#
# # Ver una versión anterior (ajusta el número tras revisar DESCRIBE HISTORY)
# spark.sql(f"SELECT * FROM {db}.adult_silver VERSION AS OF 0 LIMIT 20").show()
#
# # (Opcional) RESTORE
# spark.sql(f"RESTORE TABLE {db}.adult_silver TO VERSION AS OF <version_anterior>")


In [0]:

# OPTIMIZE + ZORDER (puede no estar disponible en Free Edition)
try:
    spark.sql(f"OPTIMIZE {db}.adult_silver ZORDER BY (education, income)")
    spark.sql(f"OPTIMIZE {db}.gold_income_by_demo ZORDER BY (education, sex)")
except Exception as e:
    print("OPTIMIZE no disponible o error:", e)


In [0]:

# VACUUM con DRY RUN (retención 7 días = 168 horas)
try:
    spark.sql(f"VACUUM {db}.adult_silver RETAIN 168 HOURS DRY RUN")
except Exception as e:
    print("VACUUM error:", e)


## 6) Validaciones y consultas finales

In [0]:

spark.sql(f"SELECT * FROM {db}.gold_income_by_demo ORDER BY high_income_rate DESC, n DESC LIMIT 20").show(truncate=False)
spark.sql(f"SELECT age_bucket, education, high_income_rate FROM {db}.gold_age_education_matrix ORDER BY age_bucket, education LIMIT 50").show(truncate=False)
spark.sql(f"SELECT * FROM {db}.gold_country_profile ORDER BY n DESC LIMIT 20").show(truncate=False)


## 7) Conclusiones
**TODO:** Resume decisiones (reglas de calidad, clave de deduplicación, por qué no particionar dada la escala, ZORDER si aplica) y hallazgos analíticos (p. ej., segmentos con mayor tasa `>50K`).  
